# 1.0 Exploración inicial de los viajes de MiBici

Primer contacto con los datos, escrito de forma directa con pandas antes de organizar el código en módulos. Lo que aquí se descubre se convierte después en las reglas de `mibici/cleaning.py`; el notebook 2.0 repite el análisis llamando a esos módulos.

In [1]:
import glob
import pandas as pd

archivos = sorted(glob.glob('../data/raw/datos_abiertos_2025_*.csv'))
len(archivos)

12

## Codificación

Al abrir los archivos con UTF-8, algunos meses fallan. Se revisa mes por mes.

In [2]:
codificaciones = {}
for f in archivos:
    try:
        open(f, 'rb').read().decode('utf-8')
        codificaciones[f[-11:-4]] = 'utf-8'
    except UnicodeDecodeError:
        codificaciones[f[-11:-4]] = 'latin-1'
pd.Series(codificaciones).value_counts()

utf-8      9
latin-1    3
Name: count, dtype: int64

In [3]:
partes = []
for f in archivos:
    d = pd.read_csv(f, encoding=codificaciones[f[-11:-4]], dtype=str, keep_default_na=False)
    d.columns = ['viaje_id', 'usuario_id', 'genero', 'anio_nacimiento', 'inicio', 'fin', 'origen_id', 'destino_id']
    partes.append(d)
df = pd.concat(partes, ignore_index=True)
print(df.shape)
df.head()

(4532032, 8)


,viaje_id,usuario_id,genero,anio_nacimiento,inicio,fin,origen_id,destino_id
0,37162342,601273,M,1982,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395
1,37162343,1571524,M,2003,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260
2,37162344,1435200,F,1984,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260
3,37162345,126327,M,1988,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11
4,37162346,416121,M,1994,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8


## Valores nulos y texto 'NULL'

El portal escribe los nulos como texto.

In [4]:
pd.DataFrame({'NULL': (df == 'NULL').sum(), 'NA': (df == 'NA').sum(), 'vacío': (df == '').sum()})

,NULL,NA,vacío
viaje_id,0,0,0
usuario_id,0,0,0
genero,40789,0,0
anio_nacimiento,0,1105,0
inicio,0,0,0
fin,0,0,0
origen_id,0,0,0
destino_id,0,0,0


## Año de nacimiento

In [5]:
anio = pd.to_numeric(df['anio_nacimiento'], errors='coerce')
print('no numéricos:', anio.isna().sum())
print('antes de 1930:', (anio < 1930).sum(), ' después de 2010:', (anio > 2010).sum())
anio.value_counts().sort_index().loc[1966:1974]

no numéricos: 1105
antes de 1930: 143  después de 2010: 398


anio_nacimiento
1966.0    20307
1967.0    30247
1968.0    32934
1969.0    36766
1970.0    91466
1971.0    36663
1972.0    36806
1973.0    34527
1974.0    41457
Name: count, dtype: int64

El año 1970 tiene más del doble de viajes que sus vecinos. Al contar usuarios distintos pasa lo mismo, lo que sugiere un valor por omisión en el registro.

In [6]:
u = df.drop_duplicates('usuario_id')
pd.to_numeric(u['anio_nacimiento'], errors='coerce').value_counts().sort_index().loc[1968:1972]

anio_nacimiento
1968.0    138
1969.0    165
1970.0    721
1971.0    171
1972.0    202
Name: count, dtype: int64

## Duración de los viajes

In [7]:
inicio = pd.to_datetime(df['inicio'])
fin = pd.to_datetime(df['fin'])
dur = (fin - inicio).dt.total_seconds() / 60
dur.describe(percentiles=[.001, .01, .05, .5, .95, .99, .999]).round(2)

count    4532032.00
mean          11.15
std           10.32
min            0.00
0.1%           0.03
1%             0.28
5%             2.47
50%            9.62
95%           24.27
99%           32.75
99.9%         56.43
max        11500.77
dtype: float64

In [8]:
mismo = df['origen_id'] == df['destino_id']
tramos = [(0, 1), (1, 2), (2, 5), (5, 30), (30, 180), (180, 1e9)]
pd.DataFrame([{'tramo_min': f'({a}, {b}]', 'misma_estacion': ((dur > a) & (dur <= b) & mismo).sum(), 'distinta_estacion': ((dur > a) & (dur <= b) & ~mismo).sum()} for a, b in tramos])

,tramo_min,misma_estacion,distinta_estacion
0,"(0, 1]",109121,5881
1,"(1, 2]",14173,40728
2,"(2, 5]",13236,665441
3,"(5, 30]",110246,3501292
4,"(30, 180]",7309,61371
5,"(180, 1000000000.0]",90,364


Los viajes de menos de dos minutos que regresan a la misma estación son, en su mayoría, bicicletas que se sacan y se devuelven de inmediato (arranques fallidos), no demanda real. Con distinta estación, menos de un minuto no alcanza para recorrer la distancia entre estaciones.

In [9]:
q1, q3 = dur.quantile([.25, .75])
print('límite superior por IQR:', round(q3 + 1.5 * (q3 - q1), 1), 'min')
print('viajes sobre ese límite:', (dur > q3 + 1.5 * (q3 - q1)).sum())
print('viajes de más de 180 min:', (dur > 180).sum())

límite superior por IQR: 28.5 min
viajes sobre ese límite: 91809
viajes de más de 180 min: 454


El criterio IQR marcaría como atípicos cerca de 92 mil viajes de más de 28.5 minutos, que son plausibles en bicicleta; por eso se usa un umbral de dominio de 180 minutos.

## Estaciones

In [10]:
st = pd.read_csv('../data/raw/nomenclatura_2026_09.csv')
ids = set(st['id'].astype(str))
print('estaciones en catálogo:', len(st), ' usadas en 2025:', df['origen_id'].nunique())
print('origen fuera del catálogo:', (~df['origen_id'].isin(ids)).sum())
print('capacidad 0:', (st['dpcapacity'] == 0).sum())

estaciones en catálogo: 484  usadas en 2025: 371
origen fuera del catálogo: 0
capacidad 0: 4


## Qué pasa a los módulos

- Detección de codificación por archivo: `dataset.detect_encoding`.
- Reglas de nulos, años, duraciones y estaciones: `cleaning.TripCleaner`.
- Umbrales en `cleaning.CleaningRules`, con una variante estricta y una laxa para compararlas en MLflow.